# Temporal-Predictive Reconstructor — Correcting Ahead of Loop Latency

This notebook implements `Ideas/05-temporal-predictive-reconstructor.md`: a reconstructor that looks at the last `N` frames from a single WFS and predicts the modal DM-command coefficients that would ideally correct the wavefront `M` frames ahead, instead of the usual "one frame in, one correction for right now out" mapping every other reconstructor in this repo uses. The goal is to see whether giving a reconstructor a short temporal window -- rather than a single instantaneous frame -- helps it get ahead of the AO loop's own structural latency (the leaky integrator's own ~2-iteration lag between a WFS measurement and its correction actually showing up in the applied DM shape; see the idea doc for why this is *not* the same thing as `LoopParams["delayFrames"]`).

**The experiment:** train three reconstructors under otherwise identical data/loss/optimizer conditions, all reading off the exact same shared sequence of drawn WFS frames/ground-truth OPDs at every training step (drawn once per step, not redrawn per architecture -- see "Sequential windowed training loop" below):
1. **Baseline (`N=1`, `M=0`)** -- a single-frame, zero-order reconstructor with no temporal window and no prediction target beyond the current tick. A network with no frame history has no basis for predicting anything but the present, so this is a true "no temporal information, no prediction" baseline -- the same kind of reactive, current-frame reconstructor every other network in this repo already is -- rather than a same-horizon comparison point.
2. **Channel-stack (`N=10`, `M=1`)** -- the `N` frames concatenated along the channel axis into a plain 2D CNN, predicting `M` frames ahead.
3. **Temporal (`N=10`, `M=1`)** -- a small Conv3D-stem network that explicitly processes the `N` frames in sequence before collapsing time, also predicting `M` frames ahead.

All three are then dropped into the *same* closed-loop leaky-integrator rollout (their output treated as "the current best command," exactly like every other reconstructor in this repo) and compared on identical, seeded atmosphere realizations. Training itself is open-loop supervised (an uncorrected atmosphere) -- see the idea doc's "Known risks" section for why this is the deliberately simpler starting point rather than full closed-loop BPTT.

This is a single, fictional demo instrument (a modulated Pyramid, nominal geometry, no bench calibration), not combined with `Ideas/04`'s dual-sensor fusion.

In [1]:
from mmengine import Config
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import os
from collections import deque
from tqdm import tqdm

from AI4AO import PyramidWFS, PhaseDataset, FramePreprocess, DeformableMirror, imshow_multiple
from AI4AO.LossFunctions import LogResidualVarianceLoss

device = 'cuda'  # set to "cpu" if CUDA is not available

## Loading the shared configuration

`TemporalPredictiveReconstructor_params.py` holds a single `WFSParams` (Pyramid, `Wavelength`/`Modulation` included directly -- there is only one sensor here, unlike `DualSensorFusion_params.py`'s per-sensor split), one `AtmosParams`/`LoopParams`/`DMParams`, and a `TrainParams` extended with `N` (temporal window length) and `M` (frames-ahead horizon). `LoopParams` is only used later by the closed-loop *evaluation* rollout (gain/leak draws) -- training itself is open-loop and never touches it.

In [2]:
paramfile = 'TemporalPredictiveReconstructor_params.py'

WFSParams = Config.fromfile(paramfile)['WFSParams']
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
DMParams = Config.fromfile(paramfile)['DMParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

N = TrainParams['N']
M = TrainParams['M']

# LogResidualVarianceLoss needs a wavelength to convert the shared residual OPD to phase.
loss_wavelength = WFSParams["Wavelength"]

PATH = "../../Data/TemporalPredictiveReconstructor/"
os.makedirs(PATH, exist_ok=True)

## The shared DM, WFS, and command basis

The DM and WFS are frozen throughout via `.eval()` alone -- only the reconstructors' weights are ever trained here. `WFS` (`TorchPropagator.py`) and `DeformableMirror` both override `train(mode)` to call `self.requires_grad_(False)` whenever `mode=False` (and `nn.Module.eval()` is defined as `self.train(False)`), so `.eval()` already freezes gradients on both -- an extra explicit `requires_grad_(False)` call would be redundant, not additionally load-bearing. `DMParams['Nmodes']` is read back from `dm.totalAct` after construction and used as the number of Zernike modes for `dm.MakeZernikeM2C()` (this repo's modal-basis builder; there is no true Karhunen-Loeve basis).

In [3]:
dm = DeformableMirror(WFSParams, DMParams, device)
dm.eval()

nModes = int(dm.totalAct.item())
M2C = dm.MakeZernikeM2C(nModes=nModes)
z_inv = torch.linalg.pinv(dm(M2C.T).flatten(start_dim=-2))

Total number of actuators: 185
Total number of actuators: 185


In [4]:
wfs = PyramidWFS(WFSParams, device)
wfs.eval()

framePreprocessor = FramePreprocess(WFSParams, wfs, device)
framePreprocessor.ProcessReference(wfs.reference_intensity)

C_wfs = wfs.pupil_centers.shape[0]  # pupil images per Pyramid frame
print(f"wfs pupils per frame: {C_wfs}, nModes: {nModes}")

wfs pupils per frame: 4, nModes: 185


c:\Users\franc\AOvenv\lib\site-packages\torch\functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\aten\src\ATen\native\TensorShape.cpp:4217.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


## Reconstructor architectures

Three reconstructors share the same encoder/head (`make_encoder_head`, reused verbatim from `DualSensorFusion.ipynb`'s `PupilCNN`) so they differ only in how they fuse the temporal window, not in downstream capacity:

- **`PupilCNN`** (identical to `DualSensorFusion.ipynb`'s class) -- used for both the `N=1` baseline and the `N=10` channel-stack architecture, just handed a different `n_channels`. A grouped-conv (`groups=n_channels`) stem processes each input image independently before the shared encoder/head.
- **`TemporalConv3DNet`** -- a small Conv3D stem (new architecture, no precedent elsewhere in this repo) that processes the `(B, C_wfs, N, H, W)` window with a grouped-`Conv3d` stem (each pupil image's own filters, same `groups` idea as `PupilCNN`, generalized to include a temporal kernel dimension), then a depthwise `Conv3d(kernel_size=(N,1,1))` that fully collapses the temporal axis, before the same shared encoder/head.

**Caveat worth flagging plainly:** because `PupilCNN`'s stem width scales with `n_channels` (`8×`/`16×` filters *per input channel*), the channel-stack architecture (`n_channels = N*C_wfs = 40`) ends up with a substantially larger stem (and total parameter count) than the `N=1` baseline (`n_channels = C_wfs = 4`) or `TemporalConv3DNet` (whose grouping is still by `C_wfs`, not `N*C_wfs`) -- this mirrors `DualSensorFusion.ipynb`'s own baseline-A/fused comparison (4 vs 5 channels), just more extreme here (4 vs 40). So if channel-stack outperforms the baseline, part of that could be "more capacity" rather than purely "temporal information" -- the parameter counts are printed below so this is transparent rather than hidden.

In [ ]:
def make_encoder_head(in_channels, Nmodes):
    encoder = nn.Sequential(
        nn.Conv2d(in_channels, 64, kernel_size=5, padding=2),
        nn.GELU(),
        nn.MaxPool2d(2),

        nn.Conv2d(64, 128, kernel_size=3, padding=1),
        nn.GELU(),
        nn.MaxPool2d(2),

        nn.Conv2d(128, 256, kernel_size=3, padding=1),
        nn.GELU(),
        nn.AdaptiveAvgPool2d(1),
    )
    head = nn.Sequential(
        nn.Flatten(),
        nn.Linear(256, Nmodes),
    )
    return encoder, head


class PupilCNN(nn.Module):
    def __init__(self, n_channels, Nmodes, multiplier = 8):
        super().__init__()

        self.stem = nn.Sequential(
            # Process each pupil image independently
            nn.Conv2d(n_channels, 8 * n_channels, kernel_size=11, padding=5),#, groups=n_channels),
            nn.GELU(),

            nn.Conv2d(8 * n_channels, 16 * n_channels, kernel_size=7, padding=3),#, groups=n_channels),
            nn.GELU(),

            nn.MaxPool2d(2),
        )
        self.encoder, self.head = make_encoder_head(16 * n_channels, Nmodes)

    def forward(self, x):
        x = self.stem(x)
        x = self.encoder(x)
        return self.head(x)


class TemporalConv3DNet(nn.Module):
    def __init__(self, n_wfs_channels, N, Nmodes):
        super().__init__()

        self.stem = nn.Sequential(
            # Process each pupil image independently, with a small (3-frame) temporal receptive field
            nn.Conv3d(n_wfs_channels, 8 * n_wfs_channels, kernel_size=(3, 11, 11), padding=(1, 5, 5)),#, groups=n_wfs_channels),
            nn.GELU(),

            nn.Conv3d(8 * n_wfs_channels, 16 * n_wfs_channels, kernel_size=(3, 7, 7), padding=(1, 3, 3)),#, groups=n_wfs_channels),
            nn.GELU(),

            nn.MaxPool3d((1, 2, 2)),  # spatial pooling only, temporal axis untouched
        )
        # Depthwise, fully collapses the temporal axis (kernel spans all N frames)
        self.temporal_collapse = nn.Conv3d(16 * n_wfs_channels, 16 * n_wfs_channels, kernel_size=(N, 1, 1))#, groups=16 * n_wfs_channels)
        self.encoder, self.head = make_encoder_head(16 * n_wfs_channels, Nmodes)

    def forward(self, x):  # x: (B, C_wfs, N, H, W)
        x = self.stem(x)
        x = self.temporal_collapse(x).squeeze(2)
        x = self.encoder(x)
        return self.head(x)


def assemble_channel_stack(window_frames):
    return torch.cat(window_frames, dim=1)


def assemble_temporal_stack(window_frames):
    return torch.stack(window_frames, dim=2)

## Sequential windowed training loop

Training is open-loop supervised: `dataset.generateClosedLoop` stays at its default `False` (no DM correction is ever applied while building a training window, so `residual_opd == opd_gt` directly), and the target is the ideal modal correction for the ground-truth OPD `M` frames after the window ends -- available for free by continuing to index the *same* `PhaseDataset` instance, since consecutive `dataset[idx]` calls advance one continuously-evolving atmosphere (Taylor frozen flow).

**This must call `dataset[idx]` sequentially from `idx=0` through `idx=N+M-1`, every episode, with no skipped or repeated indices** -- two `PhaseDataset` gotchas make this non-negotiable:
- `translationPhase` (the per-tick wind-translation phasor) only ever gets computed inside the `idx==1` branch of `PhaseDataset.GetMovingWavefront` -- skip that index and the atmosphere silently stops evolving (`translationPhase` stays `1.`), with no error, just a frozen screen.
- `idx==0` redraws a brand-new random atmosphere (`ResetMovingWavefront` + `DrawRandomParameters`) -- calling it a second time mid-episode would silently splice two unrelated atmosphere realizations into one training example.

`Trainer.train()` can't be reused here: beyond only accepting a single WFS frame per call, its entire leaky-integrator BPTT bookkeeping targets "the correction for right now," not a future ground truth, so this loop is hand-rolled (structurally simpler than `Trainer.train()` in that respect, but novel in needing a frame-history buffer `Trainer.train()` has no concept of).

Two versions of this loop are defined below. `train_temporal` trains one reconstructor at a time -- kept because the `M`-sweep section further down needs to retrain a single architecture across several `M` values, one at a time. `train_all_temporal` trains the baseline (`N=1`, `M=0`), channel-stack, and temporal reconstructors *together*, from one shared sequential draw per step: the baseline's single frame/target and the other two's `N`-frame window/`t+M` target are all just different slices of the same `idx=0..N+M-1` sequence (the baseline only ever needs `idx=0`, which every draw visits first regardless of `N`/`M`), so there is no reason to redraw the atmosphere and re-run the WFS propagation three separate times to train all three reconstructors.

In [6]:
def train_temporal(N, M, dataset, phaseReconstructor, optimizer, loss, assemble_fn, training_steps):
    dm.eval()
    wfs.eval()
    phaseReconstructor.train()

    M2C_T = M2C.T
    loss_tracker = torch.zeros(training_steps, device=device)
    progressBar = tqdm(range(training_steps))

    for u in progressBar:
        with torch.no_grad():
            window_frames = []
            for idx in range(N + M):  # idx = 0 .. N+M-1, strictly sequential -- see markdown above
                batch = dataset[idx]  # idx==0 draws a fresh atmosphere for this episode
                opd_gt = batch["opd"]
                pupilGT = batch["pupil"]

                if idx == 0:
                    wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

                residual_opd = opd_gt  # open loop: no DM correction is ever applied
                wfs_frame = wfs(residual_opd, pupilGT)
                preprocessed = framePreprocessor.ProcessFrame(wfs_frame)
                if idx < N:
                    window_frames.append(preprocessed)  # ticks 0 .. N-1 form the window

            target_opd, target_pupil = opd_gt, pupilGT  # last idx visited (N+M-1) is the t+M target
            window = assemble_fn(window_frames)
            Ze_future = torch.matmul(target_opd.flatten(start_dim=-2), z_inv)

        z_pred = phaseReconstructor(window)
        opd_reconstructed_pred = dm(z_pred @ M2C_T)
        corrected_residual_opd = target_opd - opd_reconstructed_pred
        loss_value = loss(Ze_future, z_pred, target_pupil, target_opd, corrected_residual_opd, wfs_frame)

        optimizer.zero_grad(set_to_none=True)
        loss_value.backward()
        optimizer.step()

        loss_tracker[u] = loss_value.detach()
        if u % 300 == 1:
            lower_lim = max(0, u - 100)
            progressBar.set_postfix({'Loss': float(loss_tracker[lower_lim:u].mean())})

    return loss_tracker


def plot_loss(loss_tracker, smoothing_window=100, title=None):
    losses = loss_tracker.detach().cpu().numpy()
    smoothed = np.convolve(losses, np.ones(smoothing_window) / smoothing_window, "valid")
    fig, ax = plt.subplots()
    ax.plot(smoothed)
    ax.set_xlabel("Iteration")
    ax.set_ylabel(r"Loss = $\ln(\mathrm{var}(\phi - \phi_{est}))$")
    if title:
        ax.set_title(title)
    plt.show()


def save_checkpoint(path, model, optimizer):
    torch.save({
        "phase_reconstructor_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
    }, path)


def load_checkpoint(path, model, optimizer, load_optimizer=True):
    if not os.path.exists(path):
        print(f"No checkpoint found at {path}, starting from scratch")
        return
    checkpoint = torch.load(path, map_location=device)
    model.load_state_dict(checkpoint["phase_reconstructor_state_dict"])
    if load_optimizer and "optimizer_state_dict" in checkpoint:
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

In [7]:
def train_all_temporal(N, M, dataset, model_baseline, model_channel, model_temporal,
                        optimizer_baseline, optimizer_channel, optimizer_temporal,
                        loss_baseline, loss_channel, loss_temporal, training_steps):
    """Trains all three reconstructors from ONE shared sequential draw per outer step.
    The baseline is a true N=1/M=0 reconstructor (its single input frame and its own
    target are both the tick at idx=0); the channel-stack and temporal networks use the
    full N-frame window and the idx=N+M-1 target. All three read off different slices of
    the exact same idx=0..N+M-1 sequence, so the atmosphere/WFS propagation is only paid
    for once per step instead of three times."""
    dm.eval()
    wfs.eval()
    model_baseline.train()
    model_channel.train()
    model_temporal.train()

    M2C_T = M2C.T
    loss_tracker_baseline = torch.zeros(training_steps, device=device)
    loss_tracker_channel = torch.zeros(training_steps, device=device)
    loss_tracker_temporal = torch.zeros(training_steps, device=device)

    progressBar = tqdm(range(training_steps))

    for u in progressBar:
        with torch.no_grad():
            window_frames = []
            for idx in range(N + M):  # idx = 0 .. N+M-1, strictly sequential -- see markdown above
                batch = dataset[idx]  # idx==0 draws a fresh atmosphere for this episode
                opd_gt = batch["opd"]
                pupilGT = batch["pupil"]

                if idx == 0:
                    wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

                residual_opd = opd_gt  # open loop: no DM correction is ever applied
                wfs_frame = wfs(residual_opd, pupilGT)
                preprocessed = framePreprocessor.ProcessFrame(wfs_frame)

                if idx == 0:
                    # Baseline (N=1, M=0): the same tick is both its only input frame and its own target
                    baseline_frame = preprocessed
                    baseline_target_opd, baseline_target_pupil, baseline_wfs_frame = opd_gt, pupilGT, wfs_frame
                if idx < N:
                    window_frames.append(preprocessed)  # ticks 0 .. N-1 form the shared N-frame window

            target_opd, target_pupil = opd_gt, pupilGT  # last idx visited (N+M-1) is the shared t+M target
            window_channel = assemble_channel_stack(window_frames)
            window_temporal = assemble_temporal_stack(window_frames)

            Ze_baseline = torch.matmul(baseline_target_opd.flatten(start_dim=-2), z_inv)
            Ze_future = torch.matmul(target_opd.flatten(start_dim=-2), z_inv)

        # --- baseline: current-frame reactive reconstruction (M=0) ---
        z_pred_baseline = model_baseline(baseline_frame)
        corrected_residual_baseline = baseline_target_opd - dm(z_pred_baseline @ M2C_T)
        loss_value_baseline = loss_baseline(Ze_baseline, z_pred_baseline, baseline_target_pupil, baseline_target_opd, corrected_residual_baseline, baseline_wfs_frame)

        optimizer_baseline.zero_grad(set_to_none=True)
        loss_value_baseline.backward()
        optimizer_baseline.step()

        # --- channel-stack (N, M) ---
        z_pred_channel = model_channel(window_channel)
        corrected_residual_channel = target_opd - dm(z_pred_channel @ M2C_T)
        loss_value_channel = loss_channel(Ze_future, z_pred_channel, target_pupil, target_opd, corrected_residual_channel, wfs_frame)

        optimizer_channel.zero_grad(set_to_none=True)
        loss_value_channel.backward()
        optimizer_channel.step()

        # --- temporal Conv3D (N, M) ---
        z_pred_temporal = model_temporal(window_temporal)
        corrected_residual_temporal = target_opd - dm(z_pred_temporal @ M2C_T)
        loss_value_temporal = loss_temporal(Ze_future, z_pred_temporal, target_pupil, target_opd, corrected_residual_temporal, wfs_frame)

        optimizer_temporal.zero_grad(set_to_none=True)
        loss_value_temporal.backward()
        optimizer_temporal.step()

        loss_tracker_baseline[u] = loss_value_baseline.detach()
        loss_tracker_channel[u] = loss_value_channel.detach()
        loss_tracker_temporal[u] = loss_value_temporal.detach()

        if u % 300 == 1:
            lower_lim = max(0, u - 100)
            progressBar.set_postfix({
                'L_base': float(loss_tracker_baseline[lower_lim:u].mean()),
                'L_chan': float(loss_tracker_channel[lower_lim:u].mean()),
                'L_temp': float(loss_tracker_temporal[lower_lim:u].mean()),
            })

    return loss_tracker_baseline, loss_tracker_channel, loss_tracker_temporal

## Closed-loop evaluation

`evaluate_temporal` generalizes `Trainer.evaluate()`: the ordinary leaky-integrator rollout (`z_estimated`/`z_buffer`/`opd_reconstructed`, unchanged -- the trained network's output is treated as "the current best command" exactly like every other reconstructor) but with a rolling buffer of the last `N` preprocessed frames instead of a single frame. For the first `N-1` ticks the buffer isn't full yet, so `z_output` is held at zero (matching `Trainer.evaluate()`'s own tolerance of a startup transient, which only engages the integrator after `i > n_steps * 0.3`) -- the assertion below checks that the buffer fills well before that warm-up period ends, so it doesn't contaminate the reported steady-state comparison.

In [8]:
assert N - 1 < TrainParams['EvalSteps'] * 0.3, "buffer fill period must stay inside the integrator warm-up window"


@torch.no_grad()
def evaluate_temporal(N, phaseReconstructor, assemble_fn, dataset, n_steps):
    dm.eval()
    wfs.eval()
    phaseReconstructor.eval()

    M2C_T = M2C.T

    batch = dataset[0]
    opd_gt = batch["opd"]
    pupilGT = batch["pupil"]
    gain = batch["loop_gain"]
    leak = batch["loop_leak"]
    wfs.SetPhotonsAndRON(batch["nphotons"], batch["ron"])

    z_estimated = torch.zeros(opd_gt.shape[0], nModes, device=device)
    z_buffer = torch.zeros_like(z_estimated)
    z_output = torch.zeros_like(z_estimated)
    opd_reconstructed = torch.zeros_like(opd_gt)

    frame_buffer = deque(maxlen=N)
    residuals, wfs_frames_list = [], []

    for i in range(n_steps):
        if i > 0:
            batch = dataset[i]
            opd_gt = batch["opd"]
            pupilGT = batch["pupil"]

        residual_opd = opd_gt - opd_reconstructed

        wfs_frame = wfs(residual_opd, pupilGT)
        preprocessed = framePreprocessor.ProcessFrame(wfs_frame, False)
        frame_buffer.append(preprocessed)

        if len(frame_buffer) == N:
            window = assemble_fn(list(frame_buffer))
            z_output = phaseReconstructor(window)
        # else: buffer not full yet, hold z_output at its previous value (zeros)

        z_buffer = torch.clone(z_output)
        if i > n_steps * 0.3:
            z_estimated = z_estimated * leak + gain * z_buffer

        opd_reconstructed = dm(z_estimated @ M2C_T)

        residuals.append(residual_opd)
        wfs_frames_list.append(wfs_frame)

    return torch.stack(residuals), torch.stack(wfs_frames_list)


def residual_rms_nm(residual_opd, pupil, steady_state_frac=0.5):
    """RMS wavefront error (nm) over the pupil, averaged over the steady-state (post loop-closure) tail of the rollout."""
    n_steps = residual_opd.shape[0]
    tail = residual_opd[int(n_steps * steady_state_frac):]
    rms_per_step = torch.sqrt(torch.mean(tail[..., pupil.bool()] ** 2, dim=-1))
    return (rms_per_step.mean() * 1e9).item()

## Training the baseline, channel-stack, and temporal reconstructors together

All three reconstructors are trained from the exact same sequence of drawn WFS frames and ground-truth OPDs at every step -- `train_all_temporal` runs the shared `idx=0..N+M-1` sequential draw once per outer step, and each reconstructor reads off whichever slice of it it needs: the baseline uses only the frame/target at `idx=0` (`M=0`), the channel-stack and temporal networks use the full `N`-frame window and the `idx=N+M-1` target. Each network still has its own optimizer and its own independent backward pass -- they just no longer pay for three separate atmosphere draws and WFS propagations to get there.

In [9]:
dataset_shared = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
dataset_shared.generateClosedLoop = False  # open-loop supervised training

CNN_baseline = PupilCNN(n_channels=C_wfs, Nmodes=nModes).to(device=device)
CNN_channel = PupilCNN(n_channels=N * C_wfs, Nmodes=nModes).to(device=device)
CNN_temporal = TemporalConv3DNet(n_wfs_channels=C_wfs, N=N, Nmodes=nModes).to(device=device)

for name, model in [("Baseline (N=1, M=0)", CNN_baseline),
                     (f"Channel-stack (N={N}, M={M})", CNN_channel),
                     (f"Temporal Conv3D (N={N}, M={M})", CNN_temporal)]:
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name} -- total trainable parameters: {total_params:,}")

optimizer_baseline = torch.optim.AdamW(CNN_baseline.parameters(), TrainParams['lrn'], fused=True)
optimizer_channel = torch.optim.AdamW(CNN_channel.parameters(), TrainParams['lrn'], fused=True)
optimizer_temporal = torch.optim.AdamW(CNN_temporal.parameters(), TrainParams['lrn'], fused=True)

loss_baseline = LogResidualVarianceLoss(dataset_shared.pupil, wavelength=loss_wavelength)
loss_channel = LogResidualVarianceLoss(dataset_shared.pupil, wavelength=loss_wavelength)
loss_temporal = LogResidualVarianceLoss(dataset_shared.pupil, wavelength=loss_wavelength)

load_checkpoint(PATH + "BaselineCNN.pth", CNN_baseline, optimizer_baseline, load_optimizer=False)
load_checkpoint(PATH + "ChannelStackCNN.pth", CNN_channel, optimizer_channel, load_optimizer=False)
load_checkpoint(PATH + "TemporalConv3DCNN.pth", CNN_temporal, optimizer_temporal, load_optimizer=False)

Baseline (N=1, M=0) -- total trainable parameters: 634,969
Channel-stack (N=10, M=1) -- total trainable parameters: 13,025,593
Temporal Conv3D (N=10, M=1) -- total trainable parameters: 907,673
No checkpoint found at ../../Data/TemporalPredictiveReconstructor/BaselineCNN.pth, starting from scratch
No checkpoint found at ../../Data/TemporalPredictiveReconstructor/ChannelStackCNN.pth, starting from scratch
No checkpoint found at ../../Data/TemporalPredictiveReconstructor/TemporalConv3DCNN.pth, starting from scratch


In [ ]:
loss_tracker_baseline, loss_tracker_channel, loss_tracker_temporal = train_all_temporal(
    N, M, dataset_shared,
    CNN_baseline, CNN_channel, CNN_temporal,
    optimizer_baseline, optimizer_channel, optimizer_temporal,
    loss_baseline, loss_channel, loss_temporal,
    TrainParams['TrainRunNb']
)

In [ ]:
plot_loss(loss_tracker_baseline, title="Baseline (N=1, M=0)")
plot_loss(loss_tracker_channel, title=f"Channel-stack (N={N}, M={M})")
plot_loss(loss_tracker_temporal, title=f"Temporal Conv3D (N={N}, M={M})")

In [ ]:
save_checkpoint(PATH + "BaselineCNN.pth", CNN_baseline, optimizer_baseline)
save_checkpoint(PATH + "ChannelStackCNN.pth", CNN_channel, optimizer_channel)
save_checkpoint(PATH + "TemporalConv3DCNN.pth", CNN_temporal, optimizer_temporal)

## Ablation: does a temporal window help, and does explicit temporal structure beat channel-stacking?

All three trained reconstructors are evaluated on **identical atmosphere realizations**: `torch.manual_seed` is reset to the same value and a fresh `PhaseDataset` is built immediately before each rollout, so the wind/r0/noise draws line up exactly across the three conditions.

In [ ]:
eval_steps = TrainParams['EvalSteps']
seed = TrainParams['EvalSeed']

torch.manual_seed(seed)
eval_dataset_baseline = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
residual_baseline, _ = evaluate_temporal(1, CNN_baseline, assemble_channel_stack, eval_dataset_baseline, eval_steps)
rms_baseline = residual_rms_nm(residual_baseline, eval_dataset_baseline.pupil)

torch.manual_seed(seed)
eval_dataset_channel = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
residual_channel, _ = evaluate_temporal(N, CNN_channel, assemble_channel_stack, eval_dataset_channel, eval_steps)
rms_channel = residual_rms_nm(residual_channel, eval_dataset_channel.pupil)

torch.manual_seed(seed)
eval_dataset_temporal = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
residual_temporal, wfs_frames_temporal = evaluate_temporal(N, CNN_temporal, assemble_temporal_stack, eval_dataset_temporal, eval_steps)
rms_temporal = residual_rms_nm(residual_temporal, eval_dataset_temporal.pupil)

print(f"Baseline (N=1, M=0):          {rms_baseline:.1f} nm RMS")
print(f"Channel-stack (N={N}, M={M}):    {rms_channel:.1f} nm RMS")
print(f"Temporal Conv3D (N={N}, M={M}):  {rms_temporal:.1f} nm RMS")

In [ ]:
fig, ax = plt.subplots()
labels = ["Baseline (N=1, M=0)", f"Channel-stack (N={N}, M={M})", f"Temporal Conv3D (N={N}, M={M})"]
values = [rms_baseline, rms_channel, rms_temporal]
ax.bar(labels, values)
ax.set_ylabel("Steady-state residual wavefront error (nm RMS)")
ax.set_title("Temporal-predictive reconstructor ablation")
plt.show()

## Does `M=1` actually match the loop's own structural latency?

Training teaches each network "given the window, output the ideal correction for `t+M`" -- i.e. it implicitly assumes the output will be applied `M` ticks after the window ends. But the leaky-integrator bookkeeping in `evaluate_temporal` (mirroring `Trainer.evaluate()`) only folds a given `z_output` into `opd_reconstructed` roughly 2 ticks later, regardless of `M`. This is an open, load-bearing question from the idea doc's "Known risks" section, not something to assume away -- so here we do a small sweep of `M` (retraining the cheaper channel-stack architecture at each value, with a shorter training budget since this is exploratory) and look at which `M` actually minimizes closed-loop residual, rather than trusting the suggested `M=1` default.

In [ ]:
sweep_rms = {}

for m_value in TrainParams['MSweepValues']:
    dataset_sweep = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
    dataset_sweep.generateClosedLoop = False

    CNN_sweep = PupilCNN(n_channels=N * C_wfs, Nmodes=nModes).to(device=device)
    optimizer_sweep = torch.optim.AdamW(CNN_sweep.parameters(), TrainParams['lrn'], fused=True)
    loss_sweep = LogResidualVarianceLoss(dataset_sweep.pupil, wavelength=loss_wavelength)

    print(f"--- Training channel-stack reconstructor for M={m_value} ---")
    train_temporal(
        N, m_value, dataset_sweep, CNN_sweep, optimizer_sweep, loss_sweep,
        assemble_channel_stack, TrainParams['MSweepRunNb']
    )

    torch.manual_seed(seed)
    eval_dataset_sweep = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
    residual_sweep, _ = evaluate_temporal(N, CNN_sweep, assemble_channel_stack, eval_dataset_sweep, eval_steps)
    sweep_rms[m_value] = residual_rms_nm(residual_sweep, eval_dataset_sweep.pupil)
    print(f"M={m_value}: {sweep_rms[m_value]:.1f} nm RMS")

In [ ]:
fig, ax = plt.subplots()
ax.plot(list(sweep_rms.keys()), list(sweep_rms.values()), marker="o")
ax.set_xlabel("M (frames-ahead prediction horizon)")
ax.set_ylabel("Steady-state residual wavefront error (nm RMS)")
ax.set_title(f"Channel-stack (N={N}): closed-loop residual vs. M")
plt.show()

## Open-loop vs. closed-loop evaluation distribution

Training uses `generateClosedLoop=False` (an uncorrected atmosphere). Evaluation above also uses `generateClosedLoop=False` datasets, matching the distribution the networks were actually supervised on -- but `DualSensorFusion.ipynb`'s precedent uses `generateClosedLoop=True` for its closed-loop rollouts. This checks both regimes on the already-trained channel-stack (`M=1`) model, rather than silently picking one.

In [ ]:
torch.manual_seed(seed)
eval_dataset_open = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset_open.generateClosedLoop = False
residual_open, _ = evaluate_temporal(N, CNN_channel, assemble_channel_stack, eval_dataset_open, eval_steps)
rms_open = residual_rms_nm(residual_open, eval_dataset_open.pupil)

torch.manual_seed(seed)
eval_dataset_closed = PhaseDataset(WFSParams, AtmosParams, LoopParams, DMParams, device)
eval_dataset_closed.generateClosedLoop = True
residual_closed, _ = evaluate_temporal(N, CNN_channel, assemble_channel_stack, eval_dataset_closed, eval_steps)
rms_closed = residual_rms_nm(residual_closed, eval_dataset_closed.pupil)

print(f"generateClosedLoop=False (matches training distribution): {rms_open:.1f} nm RMS")
print(f"generateClosedLoop=True  (DualSensorFusion.ipynb's convention):  {rms_closed:.1f} nm RMS")

## Visualizing the temporal reconstructor's closed loop

A quick sanity check at steady state (last frame of the temporal architecture's ablation rollout above).

In [ ]:
fig, axes = imshow_multiple(
    [
        {"tensor": residual_temporal[-1], "title": "Residual OPD (temporal, steady state)", "same_scale": True},
        {"tensor": wfs_frames_temporal[-1], "title": "wfs frame"},
    ],
    max_channel_number=4
)
plt.show()